# Lesson 06 Walkthrough: Database Design & Normalization

**Course:** Database Applications Development  
**Lesson:** 06 - Database Design and Normalization

---

## Learning Objectives

In this walkthrough, we will:
1. Analyze the NBA database structure
2. Verify normalization of our tables
3. Identify primary and foreign keys
4. Examine relationships between tables
5. Practice identifying normalization violations
6. Fix poorly designed tables

**Skills practiced:**
- Database analysis
- Normalization principles (1NF, 2NF, 3NF)
- Table design
- Identifying relationships

---

## Setup

In [ ]:
import pandas as pd
import sqlite3

# Connect to database
conn = sqlite3.connect('nba_5seasons.db')
print("✅ Connected to NBA database")

---

## Part 1: Examining Database Structure

Let's start by looking at what tables we have and their schemas.

### Step 1.1: List All Tables

In [ ]:
# TODO: Write a query to show all tables in the database
# Hint: SELECT name FROM sqlite_master WHERE type='table'
query = """

"""

tables = pd.read_sql(query, conn)
print("Tables in our NBA database:")
print(tables)

**Question:** How many tables are in our database?

**Your answer:**

### Step 1.2: Examine teams Table Structure

In [ ]:
# TODO: Get schema for teams table using PRAGMA table_info()
query = """

"""
teams_schema = pd.read_sql(query, conn)
print("teams table schema:")
print(teams_schema)

In [ ]:
# Look at sample data
query = "SELECT * FROM teams LIMIT 5"
teams_sample = pd.read_sql(query, conn)
print("\nSample teams data:")
print(teams_sample)

**Questions to discuss:**

1. What is the primary key of the teams table?

**Your answer:**

2. Why is team_id a better primary key than full_name?

**Your answer:**

3. Are all values in the teams table atomic (indivisible)?

**Your answer:**

4. What normal form is this table in?

**Your answer:**

### Step 1.3: Examine players Table Structure

In [ ]:
# TODO: Get schema for players table
query = """

"""
players_schema = pd.read_sql(query, conn)
print("players table schema:")
print(players_schema)

In [ ]:
# Look at sample data
query = "SELECT * FROM players LIMIT 5"
players_sample = pd.read_sql(query, conn)
print("\nSample players data:")
print(players_sample)

**Critical Question:**

Why is there NO team_id column in the players table?

**Your answer:**

**Hint:** Think about what happens when a player gets traded to a different team.

### Step 1.4: Examine team_game_stats Table Structure

In [ ]:
# TODO: Get schema for team_game_stats table
query = """

"""
tgs_schema = pd.read_sql(query, conn)
print("team_game_stats table schema:")
print(tgs_schema)

In [ ]:
# Look at sample data
query = "SELECT * FROM team_game_stats LIMIT 5"
tgs_sample = pd.read_sql(query, conn)
print("\nSample team_game_stats data:")
print(tgs_sample)

**Questions to discuss:**

1. What is the primary key? Is it a single column or composite?

**Your answer:**

2. Why do we need BOTH game_id and team_id for the primary key?

**Your answer:**

3. What foreign key(s) does this table have?

**Your answer:**

4. Do all the statistics columns (pts, wl, etc.) depend on the entire primary key?

**Your answer:**

### Step 1.5: Examine player_season_stats Table Structure

In [ ]:
# TODO: Get schema for player_season_stats table
query = """

"""
pss_schema = pd.read_sql(query, conn)
print("player_season_stats table schema:")
print(pss_schema)

In [ ]:
# Look at sample data
query = "SELECT * FROM player_season_stats LIMIT 5"
pss_sample = pd.read_sql(query, conn)
print("\nSample player_season_stats data:")
print(pss_sample)

**Questions to discuss:**

1. What columns make up the primary key?

**Your answer:**

2. Why do we need ALL THREE columns (player_id, team_id, season) for uniqueness?

**Your answer:**

3. What foreign keys does this table have?

**Your answer:**

4. Is this table serving as a junction table? If so, what relationship does it create?

**Your answer:**

---

## Part 2: Identifying Relationships

Let's verify the relationships between our tables.

### Relationship 1: teams ↔ team_game_stats

In [ ]:
# TODO: Count how many games each team has in the 2021-22 season
# This will show us the 1:M relationship
query = """

"""

result = pd.read_sql(query, conn)
print("Games per team (2021-22 season):")
print(result)

**Question:** What type of relationship is this (1:1, 1:M, or M:N)?

**Your answer:**

### Relationship 2: Find players who changed teams

In [ ]:
# This query shows the M:N relationship between players and teams
query = """
SELECT 
    p.full_name,
    pss.season,
    COUNT(DISTINCT pss.team_id) as team_count
FROM players p
JOIN player_season_stats pss ON p.player_id = pss.player_id
WHERE pss.season = '2021-22'
GROUP BY p.player_id, pss.season
HAVING COUNT(DISTINCT pss.team_id) > 1
LIMIT 10
"""

result = pd.read_sql(query, conn)
print("Players who changed teams in 2021-22:")
print(result)

**Question:** This demonstrates what type of relationship between players and teams?

**Your answer:**

**Question:** What table serves as the junction table for this M:N relationship?

**Your answer:**

---

## Part 3: Normalization Practice

Let's practice identifying and fixing normalization violations.

### Example 1: Violating 1NF

**BAD Design:**
```
Players_Bad:
player_id | name          | positions
1         | LeBron James  | SF, PF
2         | Stephen Curry | PG, SG
```

In [ ]:
# Create the BAD table (for demonstration)
query = """
CREATE TABLE IF NOT EXISTS players_bad (
    player_id INTEGER PRIMARY KEY,
    name TEXT,
    positions TEXT
)
"""
conn.execute(query)

query = """
INSERT OR REPLACE INTO players_bad VALUES
    (1, 'LeBron James', 'SF, PF'),
    (2, 'Stephen Curry', 'PG, SG')
"""
conn.execute(query)
conn.commit()

print("BAD Design (violates 1NF):")
print(pd.read_sql("SELECT * FROM players_bad", conn))

**Question:** Why does this violate 1NF?

**Your answer:**

**Question:** How would you fix this design?

**Your answer:**

### Example 2: Violating 3NF

**BAD Design:**
```
Player_Stats:
player_id | player_name | team_id | team_name     | team_city
1         | LeBron      | 1610612747 | Lakers     | Los Angeles
2         | Curry       | 1610612744 | Warriors   | San Francisco
```

In [ ]:
# Create BAD table
query = """
CREATE TABLE IF NOT EXISTS player_stats_bad (
    player_id INTEGER PRIMARY KEY,
    player_name TEXT,
    team_id INTEGER,
    team_name TEXT,
    team_city TEXT
)
"""
conn.execute(query)

query = """
INSERT OR REPLACE INTO player_stats_bad VALUES
    (1, 'LeBron James', 1610612747, 'Los Angeles Lakers', 'Los Angeles'),
    (2, 'Anthony Davis', 1610612747, 'Los Angeles Lakers', 'Los Angeles'),
    (3, 'Stephen Curry', 1610612744, 'Golden State Warriors', 'San Francisco')
"""
conn.execute(query)
conn.commit()

print("BAD Design (violates 3NF):")
print(pd.read_sql("SELECT * FROM player_stats_bad", conn))

**Question:** What's wrong with this design? (Hint: Look at the Lakers players)

**Your answer:**

**Question:** What is the transitive dependency?

**Your answer:**

**Question:** What problems could this cause?

**Your answer:**

---

## Part 4: Your Analysis

Based on what you've learned, answer these summary questions:

**1. Is our NBA database well-designed? Why or why not?**

**Your answer:**

**2. What normal form are our tables in?**

**Your answer:**

**3. Why does player_season_stats need a composite primary key with three columns?**

**Your answer:**

**4. Give one advantage of our database design:**

**Your answer:**

---

## Cleanup

In [ ]:
# Clean up demonstration tables
conn.execute("DROP TABLE IF EXISTS players_bad")
conn.execute("DROP TABLE IF EXISTS player_stats_bad")
conn.commit()

# Close connection
conn.close()
print("✅ Walkthrough complete!")

---

## Next Steps

Now that you understand our NBA database structure, you're ready to:
- Complete the task notebooks
- Practice designing your own databases
- Fix poorly designed tables
- Create ERD diagrams

**Good database design makes everything else easier!**